# Business purpose — data quality and site profiles

This notebook lets an energy portfolio operator see whether each **simulated** demand or renewable site is sufficiently complete, physically plausible and interpretable for later research. It reuses Phase 4 readiness and audit artifacts; flagged observations remain present.

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
from gridmatch.research.common import project_path
from gridmatch.research.profiles import profile_summaries, save_profile_artifacts

np.random.seed(20260725)
sites = pd.read_parquet(project_path('data', 'demo', 'sites.parquet'))
observations = pd.read_parquet(project_path('data', 'demo', 'observations.parquet'))
readiness = pd.read_parquet(project_path('data', 'quality', 'site_readiness.parquet'))
annotated = pd.read_parquet(project_path('data', 'quality', 'annotated_observations.parquet'))
assert set(sites['data_origin']) == {'simulated'}
assert set(observations['data_origin']) == {'simulated'}
assert len(annotated) == len(observations)
summaries = profile_summaries(sites, observations, readiness, annotated)
artifact_paths = save_profile_artifacts(sites, observations, readiness, annotated)
display(summaries['archetypes'])
display(summaries['quality_score_distribution'])
display(summaries['readiness'][['site_id', 'quality_score', 'site_readiness']])
print({name: str(path) for name, path in artifact_paths.items()})

,site_role,business_archetype,technology,site_count
0,demand,hospitality,grid_supply,1
1,demand,manufacturing,grid_supply,1
2,demand,office,grid_supply,2
3,demand,retail,grid_supply,2
4,demand,warehouse,grid_supply,2
5,generation,renewable_generator,solar,2
6,generation,renewable_generator,wind,2


,site_count,minimum,p25,median,p75,maximum,ready_sites
0,12,95.55,99.73,99.84,99.84,99.84,12


,site_id,quality_score,site_readiness
0,dem_office_london,99.84,ready
1,dem_office_bristol,99.84,ready
2,dem_warehouse_manchester,99.84,ready
3,dem_warehouse_glasgow,99.84,ready
4,dem_retail_birmingham,99.84,ready
5,dem_retail_cardiff,99.84,ready
6,dem_hospitality_edinburgh,99.84,ready
7,dem_manufacturing_sheffield,99.84,ready
8,gen_solar_cambridge,95.55,ready
9,gen_solar_cornwall,95.57,ready


{'archetypes_table': 'C:\\Users\\Michael\\Code\\Volter\\artifacts\\tables\\02_archetypes.csv', 'half_hour_profiles_table': 'C:\\Users\\Michael\\Code\\Volter\\artifacts\\tables\\02_half_hour_profiles.csv', 'capacity_factors_table': 'C:\\Users\\Michael\\Code\\Volter\\artifacts\\tables\\02_capacity_factors.csv', 'missingness_table': 'C:\\Users\\Michael\\Code\\Volter\\artifacts\\tables\\02_missingness.csv', 'quality_flags_table': 'C:\\Users\\Michael\\Code\\Volter\\artifacts\\tables\\02_quality_flags.csv', 'quality_score_distribution_table': 'C:\\Users\\Michael\\Code\\Volter\\artifacts\\tables\\02_quality_score_distribution.csv', 'readiness_table': 'C:\\Users\\Michael\\Code\\Volter\\artifacts\\tables\\02_readiness.csv', 'profile_dem_office_london': 'C:\\Users\\Michael\\Code\\Volter\\artifacts\\figures\\02_profile_dem_office_london.png', 'profile_dem_manufacturing_sheffield': 'C:\\Users\\Michael\\Code\\Volter\\artifacts\\figures\\02_profile_dem_manufacturing_sheffield.png', 'profile_gen_sola

## Profiles, capacity factors and weekday effects

Average half-hour shapes expose archetype schedules and weekday/weekend differences. Generator capacity factors are descriptive ratios over the simulated period, not performance claims about public assets.

In [2]:
display(summaries['capacity_factors'])
display(summaries['half_hour_profiles'].head(12))
display(summaries['missingness'])

,site_id,total_generation_mwh,observed_periods,installed_capacity_mw,capacity_factor
0,gen_solar_cambridge,2512.194281,8640,4.2,0.138459
1,gen_solar_cornwall,1847.149083,8640,3.1,0.137929
2,gen_wind_aberdeenshire,7067.919377,8640,7.5,0.218146
3,gen_wind_cumbria,5680.087296,8640,6.0,0.219139


,site_id,day_type,settlement_period,mean_energy_mwh
0,dem_hospitality_edinburgh,weekday,1,0.118974
1,dem_hospitality_edinburgh,weekday,2,0.115123
2,dem_hospitality_edinburgh,weekday,3,0.114935
3,dem_hospitality_edinburgh,weekday,4,0.114255
4,dem_hospitality_edinburgh,weekday,5,0.116734
5,dem_hospitality_edinburgh,weekday,6,0.116944
6,dem_hospitality_edinburgh,weekday,7,0.117715
7,dem_hospitality_edinburgh,weekday,8,0.119227
8,dem_hospitality_edinburgh,weekday,9,0.121333
9,dem_hospitality_edinburgh,weekday,10,0.123765


,column,missing_rate
0,timestamp_utc,0.0
1,settlement_date,0.0
2,site_id,0.0
3,site_role,0.0
4,technology,0.0
5,observed_power_mw,0.0
6,energy_mwh,0.0
7,temperature_c,0.0
8,irradiance_wm2,0.0
9,wind_speed_mps,0.0


## Quality flags and readiness

The Phase 4 score combines completeness, consistency, physical validity, recency and anomaly rate. All 12 sites are currently `ready` because their scores exceed the documented threshold and no critical physical/timestamp failures occur. The 5,086 flagged rows in the current artifact are retained rule-based candidates—mostly robust spikes plus long zero runs—not automatically confirmed faults.

In [3]:
display(summaries['quality_flags'])
flagged_count = int((annotated['quality_flag'] != 'valid').sum())
print({'source_rows': len(observations), 'annotated_rows': len(annotated), 'flagged_rows': flagged_count, 'all_rows_retained': len(observations) == len(annotated)})

,quality_flag,flagged_rows
0,extreme_spike,4936
1,long_zero_run,150


{'source_rows': 103680, 'annotated_rows': 103680, 'flagged_rows': 5086, 'all_rows_retained': True}


## Findings, limitations and production implications

**Findings:** office, manufacturing, solar and wind profiles are visibly distinct; all sites pass current readiness criteria; all flags remain auditable. **Limitations:** data is simulated and the rules deliberately favour sensitivity, so a flag is not proof of bad data. **Production implications:** decide per use case whether to exclude, down-weight, repair or explicitly model flagged rows; record that decision and never overwrite the source observation.